<div>
    <img src="imgs/EII-ULPGC-logo.jpeg" width="400" align="right">
</div>

# **Fine tuning de un modelo con dataset tipo 'instruct'**



In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import wandb
import os

os.environ['WANDB_NOTEBOOK_NAME'] = 'LanguageModeling.ipynb'

# Configuración del modelo y tokenizer
# model_name = "./fine_tuned_model_autoregressive"
base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

model_name = "C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive_merged"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map='auto')

# Preparación del dataset
train_dataset = load_dataset('json', data_files='./data/questions_train.jsonl', split="train")
# valid_dataset = load_dataset('json', data_files='./data/content/test.jsonl', split="train")


def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])


wandb.init(
    project='LM',
    # name='iris-' + str(model_config['hidden_size']) + '-hidden_' + str(model_config['learning_rate']) + '-lr',
    # config=model_config
)


# Configuración de LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    r=8,  # rango de LoRA
    lora_alpha=16,  # hiperparámetro de LoRA
    lora_dropout=0.1,  # dropout de LoRA
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']  # módulos objetivo para aplicar LoRA
)

model = get_peft_model(model, lora_config)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=1e-4,
    fp16=True,
    # evaluation_strategy="steps",
    eval_steps=10_000,
    report_to="wandb"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Configuración del Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train_dataset['train'],
#     data_collator=data_collator
# )

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    # eval_dataset=valid_dataset_mapped,   
    # dataset_text_field="text",
    max_seq_length=64,
    tokenizer=tokenizer,
    args=training_args,
    # compute_metrics=compute_metrics_fn,
    packing=True,
    data_collator=data_collator,
)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, packing. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:181: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:269: Use

In [6]:
trainer.train()

Step,Training Loss
100,1.076500
200,0.524000


c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in C:\Users\NLP\Propio\Research\Notebooks\TRAINED MODELS\fine_tuned_model_autoregressive_merged - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=250, training_loss=0.7362500457763672, metrics={'train_runtime': 77.339, 'train_samples_per_second': 6.465, 'train_steps_per_second': 3.233, 'total_flos': 1444972068864000.0, 'train_loss': 0.7362500457763672, 'epoch': 10.0})

In [7]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Función para generar texto
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.01
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [10]:
# Ejemplo de uso
prompt = "<s>[INST] ¿Cuál es el teléfono de Salvador? [/INST]"
generated_texts = generate_text(prompt, max_length=45, num_return_sequences=1)

# Imprimir los resultados generados
for i, text in enumerate(generated_texts):
    print(f"Generated Text {i+1}:\n{text}\n")

Generated Text 1:
<s>[INST] ¿Cuál es el teléfono de Salvador? [/INST] El teléfono de Salvador es el 619977905. </s>

